In [ ]:
import os

os.environ.pop("TRITON_INTERPRET", None)

In [ ]:
import torch
import triton

from rl_kernels import dpo

In [ ]:
def has_triton_driver() -> bool:
    """Returns True if Triton has an active hardware driver available."""
    try:
        from triton.backends import backends

        return any(b.driver.is_active() for b in backends.values())
    except Exception:
        return False

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["N"],
        x_vals=[2**i for i in range(10, 20)],
        line_arg="provider",
        line_vals=["triton", "pytorch"],
        line_names=["Triton", "PyTorch"],
        styles=[("blue", "-"), ("green", "--")],
        ylabel="GB/s",
        plot_name="vector-add-performance",
        args={},
    )
)
def benchmark(N, provider):
    if not dpo.TRITON_AVAILABLE:
        raise RuntimeError("Requires OpenAI Triton availability.")

    device = torch.accelerator.current_accelerator()
    policy_chosen_logps = torch.randn(N, requires_grad=True, device=device)
    policy_rejected_logps = torch.randn(N, requires_grad=True, device=device)
    ref_chosen_logps = torch.randn(N, requires_grad=False, device=device)
    ref_rejected_logps = torch.randn(N, requires_grad=False, device=device)
    beta = 0.1
    label_smoothing = 0.05

    # Quantify performance by calculating execution time via triton.testing.do_bench
    quantiles = [0.5, 0.2, 0.8]

    # benchmark PyTorch baseline
    if provider == "pytorch":
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: dpo._compute_dpo_losses_torch(
                policy_chosen_logps=policy_chosen_logps,
                policy_rejected_logps=policy_rejected_logps,
                ref_chosen_logps=ref_chosen_logps,
                ref_rejected_logps=ref_rejected_logps,
                beta=beta,
                label_smoothing=label_smoothing,
            ),
            quantiles=quantiles,
        )

    # benchmark Custom Triton kernel
    if provider == "triton":
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: dpo._compute_dpo_losses_triton(
                policy_chosen_logps=policy_chosen_logps,
                policy_rejected_logps=policy_rejected_logps,
                ref_chosen_logps=ref_chosen_logps,
                ref_rejected_logps=ref_rejected_logps,
                beta=beta,
                label_smoothing=label_smoothing,
            ),
            quantiles=quantiles,
        )

    def get_gbps(ms: float) -> float:
        return 4 * policy_chosen_logps.element_size() * N / ms * 1e-6

    return get_gbps(ms), get_gbps(max_ms), get_gbps(min_ms)

In [ ]:
# run the benchmark and trigger plot saving
benchmark.run(show_plots=True, print_data=True, save_path=".")